In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna 
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline, FunctionTransformer 
import statsmodels.api as sm
from sklearn.feature_selection import SequentialFeatureSelector
#from utils.perm_class import ClassificationCV
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression, RidgeClassifier
import pyarrow



Using 10% of data (stratified & randomized) for EDA, when fitting model will use yield for the whole dataset

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

server = '.\SQLEXPRESS' 
database = 'ClubLoanData'

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    f"DRIVER={{SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

engine = create_engine(connection_url)

try:
    query = """SELECT * FROM (
    SELECT TOP 10 PERCENT * FROM dbo.loan_model_ready 
    WHERE predictor = 1 
    ORDER BY NEWID()) AS Defaults

UNION ALL

SELECT * FROM (
    SELECT TOP 10 PERCENT * FROM dbo.loan_model_ready 
    WHERE predictor = 0 
    ORDER BY NEWID()) AS GoodLoans
"""
    df = pd.read_sql(query, engine)
    df.to_csv('Data/loan_data_sample.csv')
    print(f"Data Shape: {df.shape}")
except Exception as e:
    print(f"Error: {e}")


In [ ]:

df = pd.read_csv('Data\loan_data_sample.csv')
features = ['loan_amnt', 'term', 'emp_length', 'home_ownership',
       'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'mths_since_last_major_derog',
       'application_type', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
       'total_il_high_credit_limit',
       'months_sincefrst_credit', 'public_record', 'is_consolidation',
       'region', 'is_currently_delinq', 'has_il_history']

index_sql = 'Loan_ID'
target = 'predictor'

df_features  = df[features]
df_predictor = pd.Series(df[target])

#print(df_features.shape,df_predictor.shape)

X_train, X_test, y_train,y_test = train_test_split(df_features,df_predictor,stratify=df_predictor,test_size=.2,random_state=11)

categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

Hypothesis Testing - Numerical Features

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(X_train[numerical_features].describe())

In [ ]:
nan1 = X_train[numerical_features].isnull().sum()
print(nan1)

Changing the imputed '999' back to NaN for T-Test. This is to make it more straightforward for other Col's with NaN.

In [ ]:
imputed_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq'
]

for col in imputed_cols:
    X_train[col] = X_train[col].replace(999.0, np.nan)

In [ ]:
pd.set_option('display.max_rows', None)

result=[]
for column in numerical_features:
    group1 = X_train[y_train == 0 ][column].dropna()
    group2 = X_train[y_train == 1 ][column].dropna()

    t_stat,p_val = stats.ttest_ind(group1,group2,equal_var=False)

    result.append({'Column':column,'p_val':p_val,
                   'Decision': 'Drop' if p_val > 0.05 else 'Keep'})

results = pd.DataFrame(result).sort_values(by='p_val')
print(results)

Significance of Categorical

In [ ]:
from scipy.stats import chi2_contingency

cat_result=[]
for column in categorical_features:
    contingency_table = pd.crosstab(X_train[column], y_train)
    chi2, p_val, dof, expected = chi2_contingency(contingency_table)

    cat_result.append({'Column':column,'p_val':p_val,
                   'Decision': 'Drop' if p_val > 0.05 else 'Keep'})

cat_results = pd.DataFrame(cat_result).sort_values(by='p_val')
print(cat_results)

Dropping the insignificant cols

In [ ]:
cols_to_drop = ['mths_since_last_major_derog','total_cu_tl','num_tl_30dpd','open_act_il','chargeoff_within_12_mths','is_consolidation',
                'total_bal_ex_mort','total_bal_il','total_il_high_credit_limit','num_tl_120dpd_2m']

X_train = X_train.drop(columns=cols_to_drop)


OneHotEncoding (essentially)


In [ ]:
X_encoded=pd.get_dummies(X_train[categorical_features],drop_first=True,sparse=False)
X_train = pd.concat([X_train[numerical_features],X_encoded],axis=1)

Correlation / VIF

In [ ]:
pd.set_option('display.max_rows', None)
corr = X_train.corrwith(y_train)
corr = corr.abs().sort_values(ascending=False)

corr_df= corr.to_frame(name='Correlation')
print(corr_df)